In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
import os

PROJECT_PATH = '/content/drive/MyDrive/MusProject'

print("Existe:", os.path.exists(PROJECT_PATH))
print("Contenido:", os.listdir(PROJECT_PATH))

sys.path.insert(0, PROJECT_PATH)

Mounted at /content/drive
Existe: True
Contenido: ['mus_gym', 'models_colab_v4', 'logs_colab_v4', 'train_colab.ipynb', 'MUS RL.gdoc', 'logs_ablation_v1', 'ablation_Study.ipynb', 'monitoreo_tensorboard.ipynb']


In [ ]:
import mus_gym
print("✅ mus_gym cargado correctamente")

✅ mus_gym cargado correctamente


In [ ]:
!pip install gymnasium stable-baselines3 sb3-contrib shimmy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 11.6 MB/s eta 0:00:00


In [ ]:
import gymnasium as gym
import numpy as np
import os
import random
from datetime import datetime
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback, BaseCallback
from stable_baselines3.common.env_util import make_vec_env
from sb3_contrib import RecurrentPPO

from mus_gym.mus_env import MusEnvironment
from mus_gym.mus_bots import HeuristicBot
from mus_gym.mus_enums import MusAction
from mus_gym.mus_selfplay_env import MusEnvSelfPlay

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
MODELS_DIR = os.path.join(PROJECT_PATH, "models_colab_v4")
LOGS_DIR = os.path.join(PROJECT_PATH, "logs_experimentos_hiperparametros")
LATEST_MODEL = os.path.join(MODELS_DIR, "FINAL_MUS_MODEL.zip")
OPPONENT_MODEL = os.path.join(MODELS_DIR, "FINAL_MUS_MODEL.zip") 

os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print("--- Configuración de rutas y modelos completada ---")

--- Configuración de rutas y modelos completada ---


In [ ]:
def linear_schedule(initial_value):
    def func(progress_remaining: float):
        return progress_remaining * initial_value
    return func

experiments = [
    {
        "name": "Exp_A_Baseline",
        "learning_rate": 0.0003,
        "ent_coef": 0.0,
        "clip_range": 0.2,
        "schedule": False
    },
    {
        "name": "Exp_B_LinearDecay",
        "learning_rate": 0.0003,
        "ent_coef": 0.0,
        "clip_range": 0.2,
        "schedule": True
    },
    {
        "name": "Exp_C_HighEntropy",
        "learning_rate": 0.0003,
        "ent_coef": 0.05,
        "clip_range": 0.1,
        "schedule": False
    }
]

In [ ]:
STEPS_PER_EXPERIMENT = 1_500_000 
NUM_ENVS = 16

In [ ]:
def run_ablation_study():
    print(f"--- 🔬 INICIANDO ESTUDIO DE ABLACIÓN ---")
    print(f"Oponente Fijo (Benchmark): {OPPONENT_MODEL}")

    for exp in experiments:
        exp_name = exp["name"]
        print(f"\n>>> 🧪 Ejecutando Rama: {exp_name}")

        env_kwargs = {"opponent_path": OPPONENT_MODEL}

        env = make_vec_env(MusEnvSelfPlay, n_envs=NUM_ENVS, vec_env_cls=SubprocVecEnv, env_kwargs=env_kwargs)

        try:
            model = RecurrentPPO.load(
                            LATEST_MODEL,
                            env=env,
                            tensorboard_log=LOGS_DIR, 
                            device="auto"
                        )
            if exp["schedule"]:
                model.lr_schedule = linear_schedule(exp["learning_rate"])
            else:
                model.lr_schedule = lambda _: exp["learning_rate"]

            for param_group in model.policy.optimizer.param_groups:
                param_group["lr"] = exp["learning_rate"]

            model.ent_coef = exp["ent_coef"]
            model.clip_range = lambda _: exp["clip_range"]

            model.learn(
                total_timesteps=STEPS_PER_EXPERIMENT,
                tb_log_name=exp_name,
                reset_num_timesteps=False
            )

            save_path = os.path.join(MODELS_DIR, f"model_{exp_name}.zip")
            model.save(save_path)
            print(f"{exp_name} finalizado y guardado.")

        except Exception as e:
            print(f"Error en la rama {exp_name}: {e}")

        finally:
            env.close()
            del model 

    print("\n--- ESTUDIO COMPLETADO ---")
    print(f"Compara las gráficas en TensorBoard apuntando a: {LOGS_DIR}")

In [9]:
if __name__ == "__main__":
    run_ablation_study()

--- 🔬 INICIANDO ESTUDIO DE ABLACIÓN ---
Oponente Fijo (Benchmark): /content/drive/MyDrive/MusProject/models_colab_v4/FINAL_MUS_MODEL.zip

>>> 🧪 Ejecutando Rama: Exp_A_Baseline
Logging to /content/drive/MyDrive/MusProject/logs_ablation_v1/Exp_A_Baseline_0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Se han truncado las últimas 5000 líneas del flujo de salida.
|    loss                 | 0.0918      |
|    n_updates            | 14360       |
|    policy_gradient_loss | -0.0232     |
|    value_loss           | 0.323       |
-----------------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 13.7         |
|    ep_rew_mean          | 0.553        |
| time/                   |              |
|    fps                  | 352          |
|    iterations           | 131          |
|    time_elapsed         | 3045         |
|    total_timesteps      | 8077312      |
| train/                  |              |
|    approx_kl            | 0.0070237974 |
|    clip_fraction        | 0.0601       |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.362       |
|    explained_variance   | -0.04        |
|    learning_rate        | 1.53e-05     |
|    loss                 | 0.0926       